In [4]:
# 1. Limpa tudo e começa do zero
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# 2. Carrega o dataset (verifique o nome exato do arquivo!)
df = pd.read_csv('campeonato-brasileiro-full.csv')
print("Linhas:", len(df))  # ← tem que dar 8134 ou 8135

# 3. Limpeza mínima
df['data'] = pd.to_datetime(df['data'], dayfirst=True)
df = df.sort_values('data').reset_index(drop=True)

# 4. Variável alvo
df['resultado'] = np.where(df['mandante_Placar'] > df['visitante_Placar'], 0,
                  np.where(df['mandante_Placar'] < df['visitante_Placar'], 2, 1))

# 5. FEATURE ENGINEERING CORRETO (esse é o que bate 58%)
teams = df['mandante'].unique().tolist() + df['visitante'].unique().tolist()
teams = list(set(teams))

# Inicializa stats
stats = {team: {'home_played':0, 'home_win':0, 'home_gf':0, 'home_ga':0,
                'away_played':0, 'away_win':0, 'away_gf':0, 'away_ga':0} for team in teams}

rows = []

for i, row in df.iterrows():
    h, a = row['mandante'], row['visitante']
    
    # Stats ATÉ o jogo anterior
    hp = max(stats[h]['home_played'], 1)
    ap = max(stats[a]['away_played'], 1)
    
    feat = {
        'h_win_rate': stats[h]['home_win'] / hp,
        'a_win_rate': stats[a]['away_win'] / ap,
        'h_gf_avg': stats[h]['home_gf'] / hp,
        'h_ga_avg': stats[h]['home_ga'] / hp,
        'a_gf_avg': stats[a]['away_gf'] / ap,
        'a_ga_avg': stats[a]['away_ga'] / ap,
        'derbi': 1 if row['mandante_Estado'] == row['visitante_Estado'] else 0
    }
    rows.append(feat)
    
    # Atualiza APÓS o jogo
    stats[h]['home_played'] += 1
    stats[a]['away_played'] += 1
    stats[h]['home_gf'] += row['mandante_Placar']
    stats[h]['home_ga'] += row['visitante_Placar']
    stats[a]['away_gf'] += row['visitante_Placar']
    stats[a]['away_ga'] += row['mandante_Placar']
    
    if row['mandante_Placar'] > row['visitante_Placar']:
        stats[h]['home_win'] += 1
    elif row['mandante_Placar'] < row['visitante_Placar']:
        stats[a]['away_win'] += 1

feat_df = pd.DataFrame(rows)
df = pd.concat([df.reset_index(drop=True), feat_df], axis=1)

# Preenche os primeiros jogos com a média geral (importantíssimo!)
for col in ['h_win_rate','a_win_rate','h_gf_avg','h_ga_avg','a_gf_avg','a_ga_avg']:
    df[col] = df[col].replace(0, df[col].mean())

# 6. Split e modelo (EXATAMENTE ASSIM)
treino = df[df['data'].dt.year <= 2019].copy()
teste  = df[df['data'].dt.year >= 2020].copy()

X_train = treino[['h_win_rate','a_win_rate','h_gf_avg','h_ga_avg','a_gf_avg','a_ga_avg','derbi',
                  'mandante_Estado','visitante_Estado']]
X_test  = teste[['h_win_rate','a_win_rate','h_gf_avg','h_ga_avg','a_gf_avg','a_ga_avg','derbi',
                 'mandante_Estado','visitante_Estado']]
y_train = treino['resultado']
y_test  = teste['resultado']

# Preprocessor
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), ['mandante_Estado','visitante_Estado']),
    ('num', StandardScaler(), ['h_win_rate','a_win_rate','h_gf_avg','h_ga_avg','a_gf_avg','a_ga_avg','derbi'])
])

X_train = preprocessor.fit_transform(X_train)
X_test  = preprocessor.transform(X_test)

# DIAGNÓSTICO FINAL — roda isso e me manda o output
print("=== VERDADEIRO TESTE SE O CÓDIGO ESTÁ CERTO ===")
print("Média das features numéricas no TREINO:")
print(preprocessor.named_transformers_['num'].mean_)
print("\nMédia das features numéricas no TESTE (depois do transform):")
print(X_test[:, -7:].mean(axis=0))   # as últimas 7 colunas são as numéricas padronizadas
print("\nDesvio padrão no TESTE:")
print(X_test[:, -7:].std(axis=0))

'''
# Modelo
model = XGBClassifier(n_estimators=500, max_depth=5, learning_rate=0.05, random_state=42, eval_metric='mlogloss')
model.fit(X_train, y_train)
pred = model.predict(X_test)

print(f"Acurácia: {accuracy_score(y_test, pred):.4f}")
print(f"Baseline (sempre mandante): {(y_test == 0).mean():.4f}")
'''

from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

# Votação simples entre HistGB e LogisticRegression
model1 = HistGradientBoostingClassifier(max_iter=600, max_depth=6, learning_rate=0.05, random_state=42)
model2 = LogisticRegression(multi_class='multinomial', max_iter=1000, class_weight='balanced')

final_model = VotingClassifier([('hgb', model1), ('lr', model2)], voting='soft')
final_model.fit(X_train, y_train)
pred = final_model.predict(X_test)
print(f"Acurácia Voting: {accuracy_score(y_test, pred):.4f}")

Linhas: 8785


c:\Users\henri\Documents\1Henrique\Repos\.brasileirao\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


=== VERDADEIRO TESTE SE O CÓDIGO ESTÁ CERTO ===
Média das features numéricas no TREINO:
[0.51701577 0.23710838 1.69380381 1.09501095 1.10462818 1.66044887
 0.10718954]

Média das features numéricas no TESTE (depois do transform):
[-0.26909454  0.03998982 -0.56689928 -0.36718875 -0.31625874 -0.58070462
 -0.03344852]

Desvio padrão no TESTE:
[0.90759157 0.69494493 0.71667137 0.56397818 0.51409397 0.7143548
 0.95600048]
Acurácia Voting: 0.4374


c:\Users\henri\Documents\1Henrique\Repos\.brasileirao\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [5]:
# EXPERIMENTO FINAL - ISSO VAI REVELAR TUDO
from sklearn.ensemble import RandomForestClassifier

# Usa só as 6 features numéricas PURAS, sem OneHot, sem nada
X_train_simple = treino[['h_win_rate','a_win_rate','h_gf_avg','h_ga_avg','a_gf_avg','a_ga_avg']]
X_test_simple  = teste[['h_win_rate','a_win_rate','h_gf_avg','h_ga_avg','a_gf_avg','a_ga_avg']]

model = RandomForestClassifier(n_estimators=500, max_depth=10, random_state=42)
model.fit(X_train_simple, y_train)
pred = model.predict(X_test_simple)

print(f"Acurácia SEM OneHot e SEM scaler: {accuracy_score(y_test, pred):.4f}")
print(f"Baseline: {(y_test == 0).mean():.4f}")

Acurácia SEM OneHot e SEM scaler: 0.4579
Baseline: 0.4584
